# Imports

In [1]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/thomas/Desktop/typiclust-cifar10/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/thomas/Desktop/typiclust-cifar10/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/thomas/Desktop/typiclust-cifar10/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py

# Config

In [2]:
config = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "data_directory": "../data",
}

augmentation_config = {
    "size": 32,
    "scale": (0.2, 1.0),
    "p_random_horizontal_flip": 0.5,
    "brightness": 0.4,
    "contrast": 0.4,
    "saturation": 0.4,
    "hue": 0.1,
    "p_color_jitter": 0.8,
    "p_grayscale": 0.2,
    "mean": (0.4914, 0.4822, 0.4465),
    "std": (0.2023, 0.1994, 0.2010),
}

training_config = {
    "epochs": 20,        # Paper uses 500; set higher for real runs.
    "batch_size": 128,   # Paper uses 512.
    "lr": 0.4,
    "momentum": 0.9,
    "weight_decay": 0.0001,
}

cluster_config = {
    "max_clusters": 500,  # Paper uses 500 for CIFAR-10/100 (Appendix F.1).
    "B": 10,              # Budget per round. B = M (10 classes) matches paper.
    "min_cluster_size": 5,  # Drop clusters smaller than this (Appendix F.1).
}

# Reproducibility

In [3]:
def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(config["seed"])

# Data Augmentation

In [4]:
class StochasticDataAugmentation:

    def __init__(self, base_transform: transforms.Compose, num_views: int = 2):
        self.base_transform = base_transform
        self.num_views = num_views

    def __call__(self, x):
        return [self.base_transform(x) for _ in range(self.num_views)]


contrastive_base_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        size=augmentation_config["size"],
        scale=augmentation_config["scale"],
    ),
    transforms.RandomHorizontalFlip(p=augmentation_config["p_random_horizontal_flip"]),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=augmentation_config["brightness"],
            contrast=augmentation_config["contrast"],
            saturation=augmentation_config["saturation"],
            hue=augmentation_config["hue"],
        ),
    ], p=augmentation_config["p_color_jitter"]),
    transforms.RandomGrayscale(p=augmentation_config["p_grayscale"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=augmentation_config["mean"], std=augmentation_config["std"]),
])

contrastive_transform = StochasticDataAugmentation(contrastive_base_transform, num_views=2)

embedding_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=augmentation_config["mean"], std=augmentation_config["std"]),
])

# SimCLR Model

In [5]:
class SimCLRModel(nn.Module):

    def __init__(self, feature_dimension: int = 128) -> None:
        super().__init__()
        self.base_encoder = torchvision.models.resnet18(weights=None)
        in_features = self.base_encoder.fc.in_features
        self.base_encoder.fc = nn.Identity()

        self.mlp = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.ReLU(),
            nn.Linear(in_features, feature_dimension),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mlp(self.base_encoder(x))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            return F.normalize(self.base_encoder(x), p=2, dim=1)


# SimCLR Loss (NT-Xent)

In [6]:
class SimCLRLoss(nn.Module):

    def __init__(self, temperature: float = 0.5) -> None:
        super().__init__()
        self.temperature = temperature

    def forward(self, z_i: torch.Tensor, z_j: torch.Tensor) -> torch.Tensor:
        N = z_i.size(0)
        z = F.normalize(torch.cat([z_i, z_j], dim=0), p=2, dim=1)  # (2N, d)

        # Cosine similarity matrix scaled by temperature
        sim = torch.matmul(z, z.T) / self.temperature            # (2N, 2N)
        # Mask diagonal (self-similarity) with a large negative value
        sim.masked_fill_(torch.eye(2 * N, device=z.device).bool(), -1e9)

        # Positive for index k is index k+N (and vice-versa)
        labels = (torch.arange(2 * N, device=z.device) + N) % (2 * N)
        return F.cross_entropy(sim, labels)

# Training Helpers

In [7]:
def similarity_statistics(z_i: torch.Tensor, z_j: torch.Tensor) -> tuple:
    with torch.no_grad():
        N = z_i.size(0)
        z = F.normalize(torch.cat([z_i, z_j], dim=0), dim=1)
        sim = torch.matmul(z, z.T)

        positives = torch.cat([torch.diag(sim, N), torch.diag(sim, -N)])
        mask = torch.eye(2 * N, device=z.device).bool()
        negatives = sim[~mask].view(2 * N, -1)

        return positives.mean().item(), negatives.mean().item()

# SimCLR Training (Step 1 of TPC_RP: Representation Learning)

In [8]:
device = torch.device(config["device"])
print(f"Using device: {device}")

model = SimCLRModel().to(device)
optimizer = optim.SGD(
    model.parameters(),
    lr=training_config["lr"],
    momentum=training_config["momentum"],
    weight_decay=training_config["weight_decay"],
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=training_config["epochs"],
    eta_min=0,
)
criterion = SimCLRLoss()

train_dataset = torchvision.datasets.CIFAR10(
    root=config["data_directory"], train=True, download=True, transform=contrastive_transform
)
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=training_config["batch_size"],
    shuffle=True,
    num_workers=2,
    persistent_workers=True,
    pin_memory=True,
    drop_last=True,
)

for epoch in range(training_config["epochs"]):
    model.train()
    total_loss = total_pos_sim = total_neg_sim = 0.0

    for views, _ in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{training_config['epochs']}"):
        x_i, x_j = views[0].to(device), views[1].to(device)
        z_i, z_j = model(x_i), model(x_j)

        pos_sim, neg_sim = similarity_statistics(z_i, z_j)
        loss = criterion(z_i, z_j)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_pos_sim += pos_sim
        total_neg_sim += neg_sim

    scheduler.step()
    n = len(train_loader)
    print(
        f"Epoch [{epoch + 1}/{training_config['epochs']}] "
        f"Loss: {total_loss / n:.4f}  "
        f"Positive similarity: {total_pos_sim / n:.3f}  "
        f"Negative similarity: {total_neg_sim / n:.3f}  "
        f"Learning rate: {scheduler.get_last_lr()[0]:.6f}"
    )

torch.save(model.state_dict(), "simclr_model.pth")

Using device: cuda
Files already downloaded and verified


/home/thomas/Desktop/typiclust-cifar10/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:82: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
Epoch 1/20:   0%|          | 0/390 [00:00<?, ?it/s]


OverflowError: Caught OverflowError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/thomas/Desktop/typiclust-cifar10/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 308, in _worker_loop
    data = fetcher.fetch(index)
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/thomas/Desktop/typiclust-cifar10/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 51, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/home/thomas/Desktop/typiclust-cifar10/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py", line 118, in __getitem__
    img = self.transform(img)
          ^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_245667/3602938488.py", line 8, in __call__
    return [self.base_transform(x) for _ in range(self.num_views)]
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/home/thomas/Desktop/typiclust-cifar10/.venv/lib/python3.12/site-packages/t

# Build Embeddings (L2-Normalised penultimate layer, 512-d)

In [ ]:
model.load_state_dict(torch.load("simclr_model.pth"))
model.eval()

embedding_dataset = torchvision.datasets.CIFAR10(
    root=config["data_directory"], train=True, download=True, transform=embedding_transform
)
embedding_loader = torch.utils.data.DataLoader(
    embedding_dataset,
    batch_size=training_config["batch_size"],
    shuffle=False,
    num_workers=2,
    persistent_workers=True,
    pin_memory=True,
    drop_last=False,
)

all_embeddings_list, all_labels_list = [], []
with torch.no_grad():
    for images, labels in tqdm(embedding_loader, desc="Embedding dataset"):
        all_embeddings_list.append(model.encode(images.to(device)).cpu())
        all_labels_list.append(labels)

all_embeddings = torch.cat(all_embeddings_list).numpy()
all_labels = torch.cat(all_labels_list).numpy()
print(f"Embeddings: {all_embeddings.shape}, Labels: {all_labels.shape}")

# Evaluation

# Baseline AL Strategies

In [ ]:
from abc import ABC, abstractmethod

class Strategy(ABC):

    @abstractmethod
    def query(self, state, budget):
        pass

## 1. Random
Selects data points entirely at random, with no consideration of model uncertainty or diversity

In [ ]:
class RandomStrategy(Strategy):

    def query(self, state, budget):
        dataset = state["dataset"]
        labeled = state["labeled"]

        pool = list(set(range(len(dataset))) - labeled)

        return random.sample(pool, budget)


## 2. Uncertainy
Chooses samples the mode is least confident about to improve learning efficiency

In [ ]:
def get_softmax_scores(
    model: nn.Module,
    dataset: torch.utils.data.Dataset,
    indices: list,
    device: torch.device,
) -> np.ndarray:
    subset = torch.utils.data.Subset(dataset, indices)
    loader = torch.utils.data.DataLoader(subset, batch_size=256, shuffle=False, num_workers=2)
    model.eval()
    probabilities = []
    with torch.no_grad():
        for images, _ in loader:
            probabilities.append(F.softmax(model(images.to(device)), dim=1).cpu().numpy())
    return np.concatenate(probabilities, axis=0)

In [ ]:
class UncertaintyStrategy(Strategy):

    def query(self, state, budget):

        model = state["model"]
        dataset = state["dataset"]
        labeled = state["labeled"]

        pool = list(set(range(len(dataset))) - labeled)
        probabilities = get_softmax_scores(model, dataset, pool, device)
        max_probabilities = probabilities.max(axis=1)
        chosen = np.argsort(max_probabilities)[:budget]

        return [pool[i] for i in chosen]

## 3. Margin
Selects examples where the difference between the top predicted classes is smallest, indicating ambiguity

In [ ]:
class MarginStrategy(Strategy):

    def query(self, state, budget):

        model = state["model"]
        dataset = state["dataset"]
        labeled = state["labeled"]

        pool = list(set(range(len(dataset))) - labeled)
        probabilities = get_softmax_scores(model, dataset, pool, device)
        sorted_probabilities = np.sort(probabilities, axis=1)
        margins = sorted_probabilities[:, -1] - sorted_probabilities[:, -2]
        chosen = np.argsort(margins)[:budget]

        return [pool[i] for i in chosen]

## 4. Entropy
Picks samples with the highest prediction uncertainty, measured by the entropy of the model's output distribution

In [ ]:
class EntropyStrategy(Strategy):

    def query(self, state, budget):

        model = state["model"]
        dataset = state["dataset"]
        labeled = state["labeled"]

        pool = list(set(range(len(dataset))) - labeled)
        probabilities = get_softmax_scores(model, dataset, pool, device)
        entropy = -np.sum(probabilities * np.log(probabilities + 1e-12), axis=1)
        chosen = np.argsort(-entropy)[:budget]

        return [pool[i] for i in chosen]

## 5. DBAL — Deep Bayesian Active Learning (Gal et al. 2017)
Uses Monte Carlo simulations to estimate prediction uncertainty and select the most informative samples


In [ ]:
def get_dropout_predictions(model, dataset, indices, device, passes=10, batch_size=256):
    model.train()
    loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, indices),
        batch_size=batch_size,
        shuffle=False
    )
    all_probabilities = []

    for _ in range(passes):
        probabilities_list = []
        with torch.no_grad():
            for x, _ in loader:
                x = x.to(device)
                logits = model(x)
                probabilities = torch.softmax(logits, dim=1)
                probabilities_list.append(probabilities.cpu().numpy())

        all_probabilities.append(np.concatenate(probabilities_list))

    return np.stack(all_probabilities)

In [ ]:
class DBALStrategy(Strategy):

    def query(self, state, budget):

        model = state["model"]
        dataset = state["dataset"]
        labeled = state["labeled"]

        pool = list(set(range(len(dataset))) - labeled)
        predictions = get_dropout_predictions(model, dataset, pool, device)

        mean_probabilities = predictions.mean(axis=0)
        entropy_mean = -np.sum(mean_probabilities * np.log(mean_probabilities + 1e-12), axis=1)
        entropy_samples = -np.sum(predictions * np.log(predictions + 1e-12), axis=2)

        mean_entropy = entropy_samples.mean(axis=0)
        mutual_info = entropy_mean - mean_entropy
        chosen = np.argsort(-mutual_info)[:budget]

        return [pool[i] for i in chosen]

## 6. CoreSet (Sener & Savarese 2018)
Chooses a diverse subset of the data that best represents the overall dataset using geometric coverage.

In [ ]:
class CoreSetStrategy(Strategy):

    def query(self, state, budget):

        embeddings = state["embeddings"]
        labeled = list(state["labeled"])
        unlabeled = list(set(range(len(embeddings))) - set(labeled))

        labeled_embeddings = embeddings[labeled]
        unlabeled_embeddings = embeddings[unlabeled]

        distance_matrix = pairwise_distances(
            unlabeled_embeddings,
            labeled_embeddings
        )

        min_distances = distance_matrix.min(axis=1)
        chosen = np.argsort(-min_distances)[:budget]

        return [unlabeled[i] for i in chosen]

## 7. BALD — Bayesian Active Learning by Disagreement (Kirsch et al. 2019)
Selects samples that maximize the expected information gain about the model parameters.

In [ ]:
class BALDStrategy(Strategy):

    def query(self, state, budget):

        model = state["model"]
        dataset = state["dataset"]
        labeled = state["labeled"]

        pool = list(set(range(len(dataset))) - labeled)
        predictions = get_dropout_predictions(model, dataset, pool, device)

        mean_probabilities = predictions.mean(axis=0)
        entropy_mean = -np.sum(mean_probabilities * np.log(mean_probabilities + 1e-12), axis=1)
        entropy_samples = -np.sum(predictions * np.log(predictions + 1e-12), axis=2)
        
        mean_entropy = entropy_samples.mean(axis=0)
        bald_score = entropy_mean - mean_entropy
        chosen = np.argsort(-bald_score)[:budget]

        return [pool[i] for i in chosen]

 ## 8. BADGE — Batch Active learning by Diverse Gradient Embeddings (Ash et al. 2020)
Selects samples whose gradients would most change the model parameters while ensuring diversity.

In [ ]:
def get_gradient_embeddings(model, dataset, indices, device, batch_size=256):

    model.eval()

    loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, indices),
        batch_size=batch_size,
        shuffle=False
    )

    embeddings = []

    with torch.no_grad():

        for x, _ in loader:

            x = x.to(device)

            logits = model(x)

            probs = torch.softmax(logits, dim=1)

            preds = probs.argmax(dim=1)

            one_hot = torch.zeros_like(probs)
            one_hot.scatter_(1, preds.unsqueeze(1), 1)

            grad = probs - one_hot

            features = model.avgpool(model.layer4(model.layer3(model.layer2(model.layer1(model.relu(model.bn1(model.conv1(x))))))))
            features = torch.flatten(features,1)

            grad_embed = torch.einsum("bi,bj->bij", grad, features)
            grad_embed = grad_embed.reshape(grad_embed.size(0), -1)

            embeddings.append(grad_embed.cpu().numpy())

    return np.concatenate(embeddings)

In [ ]:
class BADGEStrategy(Strategy):

    def query(self, state, budget):

        model = state["model"]
        dataset = state["dataset"]
        labeled = state["labeled"]

        pool = list(set(range(len(dataset))) - labeled)

        gradient_embeddings = get_gradient_embeddings(
            model,
            dataset,
            pool,
            device
        )

        kmeans = KMeans(n_clusters=budget)
        kmeans.fit(gradient_embeddings)

        centers = kmeans.cluster_centers_
        distances = pairwise_distances(centers, gradient_embeddings)
        chosen = distances.argmin(axis=1)

        return [pool[i] for i in chosen]

# TPC_RP AL Strategy

In [ ]:
class TypiclustStrategy(Strategy):
    def __init__(self, max_clusters: int = 500, min_cluster_size: int = 5):
        self.max_clusters = max_clusters
        self.min_cluster_size = min_cluster_size

    def compute_typicality(self, cluster_embeddings: np.ndarray, K: int = 20) -> np.ndarray:
        cluster_size = cluster_embeddings.shape[0]
        K = min(K, cluster_size - 1)
        if K <= 0:
            return np.ones(cluster_size, dtype=float)

        nn_model = NearestNeighbors(n_neighbors=K + 1, metric="euclidean")
        nn_model.fit(cluster_embeddings)
        distances, _ = nn_model.kneighbors(cluster_embeddings)

        mean_dist = np.mean(distances[:, 1:], axis=1)  # Exclude self-distance
        return 1.0 / (mean_dist + 1e-8)

    def get_clustering_model(self, n_clusters: int):
        if n_clusters <= 50:
            return KMeans(n_clusters=n_clusters, n_init=10)
        return MiniBatchKMeans(n_clusters=n_clusters, batch_size=1024, n_init=10)

    def eligible_clusters(self, cluster_labels: np.ndarray, labeled_clusters: set) -> list:
        eligible = [
            cluster for cluster in np.unique(cluster_labels)
            if cluster not in labeled_clusters and np.sum(cluster_labels == cluster) >= self.min_cluster_size
        ]
        if not eligible:
            # Relax the constraint if all clusters are covered
            eligible = [
                cluster for cluster in np.unique(cluster_labels)
                if np.sum(cluster_labels == cluster) >= self.min_cluster_size
            ]
        return eligible

    def select_largest_cluster(self, cluster_labels: np.ndarray, eligible: list) -> int:
        return max(eligible, key=lambda c: np.sum(cluster_labels == c))

    def query(self, state: dict, budget: int) -> list:
        embeddings = state["embeddings"]
        labeled_set = state["labeled"]

        # Determine number of clusters (K)
        n_clusters = min(len(labeled_set) + budget, self.max_clusters)
        clustering_model = self.get_clustering_model(n_clusters)
        cluster_labels = clustering_model.fit_predict(embeddings)

        # Clusters that already contain labeled samples
        labeled_cluster_ids = set(cluster_labels[i] for i in labeled_set)

        new_indices = []
        remaining = budget
        temp_labels = cluster_labels.copy()

        while remaining > 0:
            eligible = self.eligible_clusters(temp_labels, labeled_cluster_ids)
            if not eligible:
                break

            selected_cluster = self.select_largest_cluster(temp_labels, eligible)
            cluster_indices = np.where(temp_labels == selected_cluster)[0]
            cluster_embeddings = embeddings[cluster_indices]

            # Pick the most typical sample
            typicality_scores = self.compute_typicality(cluster_embeddings, K=20)
            best_local_idx = np.argmax(typicality_scores)
            chosen_idx = int(cluster_indices[best_local_idx])

            new_indices.append(chosen_idx)
            labeled_cluster_ids.add(selected_cluster)
            temp_labels[temp_labels == selected_cluster] = -1  # Mark cluster as consumed
            remaining -= 1

        return new_indices

# Classifer Training & Evaluation

In [ ]:
def build_resnet18_classifier(device: torch.device, num_classes: int = 10) -> nn.Module:
    classifier = torchvision.models.resnet18(weights=None)
    classifier.fc = nn.Linear(classifier.fc.in_features, num_classes)    
    return classifier.to(device)

In [ ]:
def train_classifier(
    labeled_indices: list,
    train_dataset: torch.utils.data.Dataset,
    device: torch.device,
    epochs: int = 100,
    batch_size: int = 64,
    model_override: nn.Module = None
) -> nn.Module:

    classifier = model_override if model_override else build_resnet18_classifier(device=device)
    
    subset = torch.utils.data.Subset(train_dataset, labeled_indices)
    loader = torch.utils.data.DataLoader(
        subset,
        batch_size=min(batch_size, len(labeled_indices)),
        shuffle=True,
        num_workers=2,
        drop_last=False
    )

    optimizer = optim.SGD(
        classifier.parameters(),
        lr=0.025,
        momentum=0.9,
        weight_decay=5e-4,
        nesterov=True
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    classifier.train()
    for _ in range(epochs):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(classifier(images), labels)
            loss.backward()
            optimizer.step()
        scheduler.step()

    return classifier

In [ ]:
def evaluate_classifier(
    classifier: nn.Module,
    test_loader: torch.utils.data.DataLoader,
    device: torch.device
) -> float:
    classifier.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = classifier(images)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return correct / total

# Active Learning Loop

In [ ]:
def add_dropout_to_resnet(model: nn.Module, p: float = 0.5) -> nn.Module:
    """
    Adds dropout layers after each residual block for MC Dropout active learning.
    """
    for name, module in model.named_modules():
        if isinstance(module, nn.Sequential):
            for i, m in enumerate(module):
                if isinstance(m, nn.ReLU):
                    module[i] = nn.Sequential(m, nn.Dropout(p=p))
    return model

In [ ]:
test_dataset = torchvision.datasets.CIFAR10(
    root=config["data_directory"], train=False, download=True, transform=embedding_transform
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=256, shuffle=False, num_workers=2
)

full_train_dataset = torchvision.datasets.CIFAR10(
    root=config["data_directory"], train=True, download=True, transform=embedding_transform
)

strategies = {
    "TPC_RP": TypiclustStrategy(
        max_clusters=cluster_config["max_clusters"],
        min_cluster_size=cluster_config["min_cluster_size"]
    ),
    "Random": RandomStrategy(),
    "Uncertainty": UncertaintyStrategy(),
    "Margin": MarginStrategy(),
    "Entropy": EntropyStrategy(),
    "DBAL": DBALStrategy(),
    "CoreSet": CoreSetStrategy(),
    "BALD": BALDStrategy(),
    "BADGE": BADGEStrategy(),
}

# Initialize labeled sets and accuracy lists for each strategy
for s in strategies.values():
    s.labeled = set()   # Track labeled indices
    s.accs = []         # Track cumulative accuracy

num_rounds = 5
B = cluster_config["B"]

# --- Active Learning Loop ---
for round_idx in range(num_rounds):
    print(f"\n=== Round {round_idx + 1}/{num_rounds} (B={B}) ===")

    for name, strat in strategies.items():
        # Build the state dictionary
        state = {
            "labeled": strat.labeled,
            "dataset": full_train_dataset,
            "embeddings": all_embeddings,  # needed by TPC_RP, CoreSet, BADGE
            "model": None                  # will be assigned if needed
        }

        # --- Cold start: random for uncertainty-based strategies ---
        if name in ["Uncertainty", "Margin", "Entropy", "DBAL", "BALD", "BADGE"] and len(strat.labeled) == 0:
            new_indices = RandomStrategy().query(state, B)
        else:
            # Train model only if the strategy requires it
            if name in ["Uncertainty", "Margin", "Entropy", "DBAL", "BALD", "BADGE"]:
                model_override = torchvision.models.resnet18(weights=None)
                model_override.fc = nn.Linear(model_override.fc.in_features, 10)
                model_override = model_override.to(device)

                # For DBAL/BALD add dropout if needed
                if name in ["DBAL", "BALD"]:
                    model_override = add_dropout_to_resnet(model_override)

                # Train classifier on current labeled set
                state["model"] = train_classifier(
                    labeled_indices=list(strat.labeled),
                    train_dataset=full_train_dataset,
                    device=device,
                    epochs=100,
                    model_override=model_override
                )

            # Query new points
            new_indices = strat.query(state, B)

        # Update labeled set
        strat.labeled.update(new_indices)

        # Train a fresh classifier for evaluation
        eval_model = train_classifier(
            labeled_indices=list(strat.labeled),
            train_dataset=full_train_dataset,
            device=device,
            epochs=100
        )
        acc = evaluate_classifier(eval_model, test_loader, device)
        strat.accs.append(acc)

        print(f"  {name:12s} | labeled={len(strat.labeled):4d} | acc={acc:.4f}")

# --- Plot Results ---
budgets = [B * (i + 1) for i in range(num_rounds)]

plt.figure(figsize=(10, 6))
for name, strat in strategies.items():
    plt.plot(budgets, [a * 100 for a in strat.accs], marker="o", label=name)
plt.xlabel("Cumulative Budget")
plt.ylabel("Test Accuracy (%)")
plt.title("CIFAR-10: Low Budget Active Learning (TPC_RP vs Baselines)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("al_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved al_results.png")

# Plot Results

In [ ]:
budgets = [B * (i + 1) for i in range(num_rounds)]

plt.figure(figsize=(10, 6))
for name, s in strategies.items():
    plt.plot(budgets, [a * 100 for a in s.accs], marker="o", label=name)
plt.xlabel("Cumulative Budget")
plt.ylabel("Test Accuracy (%)")
plt.title("CIFAR-10: Low Budget Active Learning (TPC_RP vs Baselines)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("al_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved al_results.png")